<a href="https://colab.research.google.com/github/DianaBarradasSanchez/EDPI/blob/main/l%C3%ADnea_de_espera.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Solución Analítica: Sistema de Línea de Espera ($M/M/1$)

Para un sistema con una tasa de llegada $\lambda = 4$ y una tasa de servicio $\mu = 6$, aplicando las fórmulas, los cálculos son los siguientes:

---

### 1. Factor de utilización ($\rho$)
Representa la fracción de tiempo que el servidor está ocupado.
$$\rho = \frac{\lambda}{\mu} = \frac{4}{6} \approx 0.6667$$

> **Interpretación:** El servidor está ocupado el **66.67%** del tiempo.

### 2. Probabilidad de que no haya unidades en el sistema ($P_0$)
Es la probabilidad de que el sistema esté completamente vacío.
$$P_0 = 1 - \frac{\lambda}{\mu} = 1 - \frac{4}{6} \approx 0.3333$$

> **Interpretación:** Existe un **33.33%** de probabilidad de que el servidor esté ocioso.

### 3. Número promedio de unidades en cola ($L_q$)
Es la cantidad esperada de clientes que están esperando en la fila.
$$L_q = \frac{\lambda^2}{\mu(\mu - \lambda)} = \frac{4^2}{6(6 - 4)} = \frac{16}{12} \approx 1.3333 \text{ unidades}$$

### 4. Número promedio de unidades en el sistema ($L_s$)
Incluye tanto a los clientes en la fila como al que está siendo atendido.
$$L_s = L_q + \frac{\lambda}{\mu} = 1.3333 + \frac{4}{6} = 2.0000 \text{ unidades}$$

### 5. Tiempo promedio que una unidad pasa en una cola ($W_q$)
Es el tiempo esperado de espera antes de empezar a ser atendido.
$$W_q = \frac{L_q}{\lambda} = \frac{1.3333}{4} \approx 0.3333 \text{ unidades de tiempo}$$

### 6. Tiempo promedio que una unidad pasa en el sistema ($W_s$)
Es el tiempo total desde que el cliente llega hasta que sale del sistema.
$$W_s = \frac{L_s}{\lambda} = \frac{2}{4} = 0.5000 \text{ unidades de tiempo}$$

### 7. Probabilidad de que haya $n$ unidades en el sistema ($P_n$)
Utilizando la fórmula $P_n = \left(\frac{\lambda}{\mu}\right)^n P_0$, evaluamos para los primeros casos:

* **Para $n=1$:** $P_1 = \left(\frac{4}{6}\right)^1 (0.3333) \approx 0.2222$ (**22.22%**)
* **Para $n=2$:** $P_2 = \left(\frac{4}{6}\right)^2 (0.3333) \approx 0.1481$ (**14.81%**)
* **Para $n=3$:** $P_3 = \left(\frac{4}{6}\right)^3 (0.3333) \approx 0.0987$ (**09.87%**)

In [5]:
#PROGRAMANDO EL SEUDOCÓDIGO (SIMULACIÓN)
import numpy as np
import pandas as pd

def simular_fila_espera(lam, mu, T_max):
    # --- Inicialización  ---
    t = 0           # Tiempo actual
    Na = 0          # Contador de llegadas
    Nd = 0          # Contador de salidas
    n = 0           # Clientes en el sistema

    # Generar primera llegada
    tA = np.random.exponential(1/lam)
    tD = float('inf') # Infinito si no hay nadie en servicio

    # Listas para recolectar datos de salida (A(i) y D(i) )
    llegadas = []
    salidas = []

    print(f"Iniciando simulación hasta T={T_max}...")

    # --- Bucle principal de eventos ---
    while tA <= T_max or n > 0:

        # Caso 1: Ocurre una llegada antes que una salida (y estamos dentro del tiempo T)
        if tA <= tD and tA <= T_max:
            t = tA                            # Restablecer t
            Na += 1                           # Na = Na + 1
            n += 1                            # n = n + 1
            llegadas.append(t)                # Registrar A(Na) = t

            # Generar hora de la siguiente llegada
            tA = t + np.random.exponential(1/lam)

            # Si es el único cliente, generar su tiempo de salida
            if n == 1:
                Y = np.random.exponential(1/mu)
                tD = t + Y

        # Caso 2: Ocurre una salida antes que una llegada
        elif tD < tA and tD <= T_max:
            t = tD                            # Restablecer t
            n -= 1                            # n = n - 1
            Nd += 1                           # Nd = Nd + 1
            salidas.append(t)                 # Registrar D(Nd) = t

            # Si queda gente, generar la salida del siguiente
            if n > 0:
                Y = np.random.exponential(1/mu)
                tD = t + Y
            else:
                tD = float('inf')

        # Caso 3 y 4: Se superó el tiempo T pero quedan clientes en el sistema
        elif min(tA, tD) > T_max and n > 0:
            t = tD                            # Solo procesamos salidas pendientes
            n -= 1
            Nd += 1
            salidas.append(t)

            if n > 0:
                Y = np.random.exponential(1/mu)
                tD = t + Y
            else:
                # Caso 4: n = 0, terminamos
                break
        else:
            break


    df_resultados = pd.DataFrame({
        'Cliente': range(1, Na + 1),
        'Tiempo_Llegada_A': llegadas,
        'Tiempo_Salida_D': salidas
    })

    # Tiempo total en el sistema (D - A)
    df_resultados['Tiempo_en_Sistema_W'] = df_resultados['Tiempo_Salida_D'] - df_resultados['Tiempo_Llegada_A']

    return df_resultados

# Parámetros solicitados
lambda_atencion = 4
mu_servicio = 6
tiempo_total = 100

resultados = simular_fila_espera(lambda_atencion, mu_servicio, tiempo_total)

# Mostrar las primeras filas
print("\nPrimeros 10 clientes:")
print(resultados.head(10))

print(f"Estadísticas Finales")
print(f"Total de llegadas (Na): {len(resultados)}")
print(f"Tiempo promedio en el sistema: {resultados['Tiempo_en_Sistema_W'].mean():.4f}")

Iniciando simulación hasta T=100...

Primeros 10 clientes:
   Cliente  Tiempo_Llegada_A  Tiempo_Salida_D  Tiempo_en_Sistema_W
0        1          0.166283         0.415849             0.249566
1        2          0.466180         0.859297             0.393117
2        3          0.571168         1.065915             0.494747
3        4          0.993036         1.342947             0.349911
4        5          0.993255         1.373812             0.380557
5        6          1.228019         1.379232             0.151214
6        7          1.234767         1.582794             0.348027
7        8          1.366063         1.796247             0.430184
8        9          1.992813         2.077034             0.084221
9       10          2.055522         2.578365             0.522843
Estadísticas Finales
Total de llegadas (Na): 383
Tiempo promedio en el sistema: 0.5019


## Interpretación y Comparación

A continuación se interpreta cómo se conectan el pseudocódigo que programamos y los resultados teóricos.

---

### 1. Tabla Comparativa de Resultados ($\lambda = 4, \mu = 6$)

Al ejecutar la simulación con un tiempo de ejecución largo (por ejemplo, $T = 5000$), los promedios de los datos recolectados por el código se aproximan a los valores teóricos de la siguiente manera:

| Variable / Métrica | Lógica en el Pseudocódigo (Simulación) | Fórmula Analítica (Teoría) | Valor Esperado |
| :--- | :--- | :--- | :--- |
| **Uso del Servidor ($\rho$)** | Porcentaje del tiempo total $t$ en el que el estado del sistema fue $n > 0$. | $\rho = \frac{\lambda}{\mu}$ | **0.6667** (66.67%) |
| **Probabilidad de Vacío ($P_0$)** | Porcentaje del tiempo total $t$ en el que el estado del sistema fue $n = 0$. | $P_0 = 1 - \frac{\lambda}{\mu}$ | **0.3333** (33.33%) |
| **Tiempo en el Sistema ($W_s$)** | Promedio de la lista de datos de salida: <br> `mean(D(i) - A(i))` | $W_s = \frac{1}{\mu - \lambda}$ | **0.5000** unidades de tiempo |

---

### 2. Pseudocódigo y la Teoría

* **La aleatoriedad y las tasas ($\lambda$ y $\mu$):** En las fórmulas analíticas, $\lambda = 4$ y $\mu = 6$ son tasas promedio fijas. En el pseudocódigo, cada vez que ocurre una llegada (Caso 1) o una salida (Caso 2), usamos `np.random.exponential` para generar el tiempo real. Esto significa que, aunque el promedio respeta la teoría, cada cliente experimenta un tiempo único y variable, imitando la realidad de una fila real.
* **El almacenamiento de datos:** El pseudocódigo utiliza los arreglos $A(i)$ (hora de llegada) y $D(i)$ (hora de salida). Al restar $D(i) - A(i)$ para cada cliente $i$, estamos calculando el tiempo exacto que pasó en el sistema. La media aritmética de todos estos datos guardados es la aproximación empírica al valor teórico $W_s$.
* **El efecto de las condiciones iniciales:** En el pseudocódigo empezamos con el sistema vacío ($n = 0, t = 0$). Las fórmulas analíticas, asumen que el sistema ya lleva un tiempo infinito funcionando y alcanzó el **"estado estable"**.